# 第19章 数据质量检查与清洗

建立数据质量检查流程，处理缺失、重复、异常和无效记录。

## 本章定位

本章按“概念 → 示例 → 练习”的顺序组织。代码单元格可以单独运行，也可以从上到下完整运行。

## 学习目标

- 生成质量概览
- 处理缺失值
- 识别并删除重复
- 使用规则标记异常


## 核心概念

- 先统计问题规模，再决定删除、填充或保留。
- 缺失值处理取决于业务含义，不能统一填0。
- 异常值应先标记和调查，不能机械删除。


## 示例 1：质量概览

同时查看形状、缺失、重复和类型。


In [ ]:
import numpy as np
import pandas as pd

orders = pd.DataFrame({
    "order_id": ["A1", "A2", "A2", "A3", "A4"],
    "region": ["华东", "华南", "华南", None, "华北"],
    "amount": [320.0, 880.0, 880.0, np.nan, 9800.0],
})
print("形状:", orders.shape)
print("缺失:\n", orders.isna().sum())
print("重复行:", orders.duplicated().sum())
print(orders.dtypes)


## 示例 2：缺失与重复处理

订单ID重复时需要明确保留规则。


In [ ]:
clean = orders.drop_duplicates(subset="order_id", keep="first").copy()
clean["region"] = clean["region"].fillna("未知")
median_amount = clean["amount"].median()
clean["amount"] = clean["amount"].fillna(median_amount)
print(clean)


## 示例 3：IQR异常标记

标记异常并保留原值，便于后续调查。


In [ ]:
q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
clean["is_outlier"] = clean["amount"] > upper
print("上界:", upper)
print(clean[clean["is_outlier"]])


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第19章 数据质量检查与清洗”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## Pandas 的学习主线

写数据字典 → 读取与检查 → 清洗类型和缺失值 → 选择与筛选 → 新增计算列 → 分组聚合 → 合并或透视 → 导出可复用结果

每一步都说明“一行代表什么”。处理前后记录行数、列数和关键字段；汇总前先确认分组粒度，避免得到数字却无法解释。


## 本模块练习方式

基础：完成一个字段清洗；提高：从明细表生成汇总表；挑战：处理重复、缺失和类型混乱，并写出清洗规则。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 示例 4：从明细表生成计算列

这一组例子只处理一个小问题。先运行代码，再逐行对照拆解说明。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华南", "华东"],
    "sales": [120, 150, 180],
    "cost": [80, 100, 130],
})
orders["profit"] = orders["sales"] - orders["cost"]
orders["profit_rate"] = orders["profit"] / orders["sales"]
print(orders.round(3))


### 逐步拆解

先新增一个简单指标，再基于它计算比例；拆成多列可以保留中间结果并方便检查。

建议第一次运行后只改一个输入值，再观察哪一个输出发生变化。


## 示例 5：从明细汇总到业务表

看懂上一个例子后，再观察同一主题在另一种数据或场景中的写法。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby(["region", "channel"], as_index=False)["sales"].sum()
print(summary)
print("地区合计：")
print(orders.groupby("region")["sales"].sum())


### 逐步拆解

先明确每一行的粒度，再选择 groupby 的字段；汇总表的每一行代表一个清晰的分组组合。

自我检查：如果把输入数量、类别或参数改成另一组值，代码是否仍然能运行？


## 教学实验：分组汇总与粒度

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "region": ["华东", "华东", "华南", "华南"],
    "channel": ["线上", "线下", "线上", "线下"],
    "sales": [120, 80, 150, 100],
})
summary = orders.groupby("region", as_index=False)["sales"].sum()
print(summary)
print("汇总表每一行代表一个地区")


### 第一个结果怎么读

先确认明细表一行代表一笔订单，再确认汇总表一行代表一个地区。`groupby` 的字段决定结果的粒度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
orders["sales_level"] = orders["sales"].map(
    lambda value: "高" if value >= 120 else "普通"
)
print(orders)
print(orders["sales_level"].value_counts())


### 第二个结果怎么读

第二个实验只增加一个分类列，不改变原始销售额。练习解释：什么时候应该新增列，什么时候应该直接筛选行？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：脏数据转换怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.Series(["12", "unknown", "18", ""])
converted = pd.to_numeric(raw, errors="coerce")
print("转换结果：")
print(converted)
print("无法转换的数量：", converted.isna().sum())
print("后续可以选择删除、填充或回查原始值。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

errors="coerce" 会把无法转换的值记录为缺失，适合先完成质量盘点；不要在没有统计数量前直接删除。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 常见误区

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据


## 综合练习

1. 创建含缺失和重复的客户表
2. 按客户ID去重
3. 使用中位数填充年龄并输出质量报告

请先独立完成，再点击下方“显示答案”查看参考代码。


In [ ]:
import numpy as np
import pandas as pd

customers = pd.DataFrame({
    "customer_id": ["U1", "U2", "U2", "U3"],
    "age": [28, np.nan, np.nan, 42],
    "city": ["上海", "广州", "广州", None],
})

# TODO: 按客户ID去重
clean = customers.drop_duplicates("customer_id").copy()

# TODO: 使用中位数填充年龄
clean["age"] = clean["age"].fillna(clean["age"].median())

# TODO: 填充城市缺失值为"未知"
clean["city"] = clean["city"].fillna("未知")

print(clean)
print(clean.isna().sum())


In [ ]:
import numpy as np
import pandas as pd

customers = pd.DataFrame({
    "customer_id": ["U1", "U2", "U2", "U3"],
    "age": [28, np.nan, np.nan, 42],
    "city": ["上海", "广州", "广州", None],
})
clean = customers.drop_duplicates("customer_id").copy()
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["city"] = clean["city"].fillna("未知")
print(clean)
print(clean.isna().sum())

# 自检
assert len(clean) == 3, "检查去重后行数：应该有3个唯一客户"
assert clean.isna().sum().sum() == 0, "检查缺失值：应该全部填充完成"


## 本章小结

建立数据质量检查流程，处理缺失、重复、异常和无效记录。

**迁移思考**：

1. 如果一个订单表中订单ID不重复，但同一用户有多个订单，去重时应该用什么键？
2. 为什么异常值应该先标记而不是直接删除？什么情况下可以删除异常值？


### 你已经掌握

- 生成质量概览
- 处理缺失值
- 识别并删除重复
- 使用规则标记异常


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 质量概览 | 同时查看形状、缺失、重复和类型。 | `pd.DataFrame()`、`orders.isna()`、`orders.duplicated()`、`.sum()` |
| 缺失与重复处理 | 订单ID重复时需要明确保留规则。 | `orders.drop_duplicates()`、`.copy()`、`.fillna()`、`.median()` |
| IQR异常标记 | 标记异常并保留原值，便于后续调查。 | `.quantile()`、`clean["amount"]`、`clean["is_outlier"]`、`clean[clean["is_outlier"]` |


### 需要注意

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据


### 完成检查

- [ ] 能够生成质量概览
- [ ] 能够处理缺失值
- [ ] 能够识别并删除重复
- [ ] 能够使用规则标记异常
